# k-Nearest Neighbours on ImageNet-style Images

**Goal:** run the same $k$-NN workflow on a folder of natural images, and see where the
approach breaks down at scale.

ImageNet is the large-scale image benchmark: about 1.3 million training images in 1,000
classes, with images around $469 \times 387$ pixels. It is not downloaded automatically
here, and raw-pixel $k$-NN is not a competitive ImageNet classifier in practice. The
purpose of this notebook is to show **why** that is true and what the alternatives are.

**Two ways to run it**

- **Real images.** Point `IMAGENET_ROOT` at a local ImageNet folder with one
  subdirectory per class. The notebook loads a small subset (6 classes, up to 80 images
  per class) at $32 \times 32$ pixels. This requires Pillow.
- **No local data.** If the folder is missing, the notebook builds a small synthetic
  dataset of coloured shapes with the same folder-style labels. Every cell still runs and
  the code path is identical; only the images change.

## Setup

The setup is split into two cells. The first imports the lightweight libraries and sets
the runtime options, so it finishes immediately. The second imports matplotlib and
scikit-learn; on a fresh machine it can take 10--30 seconds because matplotlib builds its
font cache once.

Loading a real image folder additionally needs Pillow (`pip install pillow`).

In [ ]:
import os
from pathlib import Path
from time import perf_counter

import numpy as np

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Core setup complete.")

The next cell is the expensive one on a first run. It imports matplotlib and
scikit-learn. The progress messages show which import is running.

In [ ]:
print("Importing matplotlib (first run may build the font cache)...", flush=True)
import matplotlib.pyplot as plt

print("Importing scikit-learn...", flush=True)
from sklearn.decomposition import PCA
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Scientific libraries ready.")

### Reusable helpers

The same helpers as in the other two notebooks, plus a gallery function for RGB images.

In [ ]:
def build_knn_pipeline(k, n_components=None, random_state=RANDOM_STATE):
    # Standardise, optionally reduce with PCA, then classify with k-NN.
    steps = [("scaler", StandardScaler())]
    if n_components is not None:
        steps.append(("pca", PCA(n_components=n_components, random_state=random_state)))
    steps.append(("knn", KNeighborsClassifier(n_neighbors=int(k), n_jobs=-1)))
    return Pipeline(steps)


def tune_k(X, y, ks, cv_splits=3, n_components=None, random_state=RANDOM_STATE):
    # Cross-validated accuracy for every candidate k.
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    rows = []
    for k in ks:
        model = build_knn_pipeline(k, n_components=n_components, random_state=random_state)
        scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
        rows.append(
            {"k": int(k), "mean": float(scores.mean()), "std": float(scores.std())}
        )
    return rows


def best_row(rows):
    return max(rows, key=lambda row: row["mean"])


def plot_k_curve(rows, title):
    ks = [row["k"] for row in rows]
    means = np.array([row["mean"] for row in rows])
    stds = np.array([row["std"] for row in rows])
    j = int(np.argmax(means))

    fig, ax = plt.subplots()
    ax.errorbar(ks, means, yerr=stds, marker="o", capsize=3)
    ax.axvline(ks[j], color="k", linestyle=":", label=f"best k = {ks[j]}")
    ax.set_xlabel("$k$")
    ax.set_ylabel("Cross-validated accuracy")
    ax.set_title(title)
    ax.legend()
    plt.show()


def fit_and_time(model, X_train, y_train, X_test, y_test):
    # Fit, predict, and report accuracy plus per-query prediction time.
    model.fit(X_train, y_train)
    model.predict(X_test[:5])  # warm-up
    start = perf_counter()
    y_pred = model.predict(X_test)
    elapsed = perf_counter() - start
    return {
        "model": model,
        "y_pred": y_pred,
        "acc": accuracy_score(y_test, y_pred),
        "seconds": elapsed,
        "ms_per_query": 1000.0 * elapsed / len(X_test),
    }


def plot_image_gallery(images, labels, class_names, shape, n=12, title=""):
    # Show a grid of grayscale or RGB images from flattened vectors.
    n = min(n, len(images))
    cols = 6
    n_rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 1.7, n_rows * 2.0))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, label in zip(axes.ravel(), images[:n], labels[:n]):
        if img.ndim == 3:
            ax.imshow(img)
        else:
            ax.imshow(img.reshape(shape), cmap="gray")
        ax.set_title(str(class_names[label]), fontsize=9)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## 1. Configure a local copy

Point the notebook at a folder of class subfolders:

```
data/imagenet/
├── n01440764/     *.JPEG   (tench)
├── n01443537/     *.JPEG   (goldfish)
├── n01484850/     *.JPEG   (great white shark)
└── ...
```

A single wrapper directory named `train`, `training`, or `images` is followed
automatically. You can also point at the validation split.

Set the path either here or with an environment variable before starting Jupyter:

```bash
export IMAGENET_ROOT=/path/to/imagenet/train
jupyter notebook Code/knn_imagenet.ipynb
```

To keep memory and time reasonable we load at most 6 classes and 80 images per class,
resized to $32 \times 32$ RGB. Change the constants below if you want a bigger sample.

In [ ]:
IMAGENET_ROOT = Path(os.environ.get("IMAGENET_ROOT", "data/imagenet")).expanduser()
IMAGE_SIZE = (32, 32)
MAX_CLASSES = 6
MAX_PER_CLASS = 80
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"ImageNet root: {IMAGENET_ROOT.resolve()}")
print(f"Folder exists: {IMAGENET_ROOT.is_dir()}")
print(f"Loading at most {MAX_CLASSES} classes x {MAX_PER_CLASS} images "
      f"at {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} pixels")

## 2. Folder loader and synthetic fallback

The loader is intentionally small: it discovers class folders, takes a random subset,
resizes each image with Pillow, and returns a flattened float array in $[0,1]$.

If no real folder is available, `make_synthetic_images` draws six classes of coloured
shapes (disk, square, triangle, ring, stripes, checkerboard) with randomised position,
size, brightness, a distractor patch, and noise. The images deliberately overlap enough
that the task is not trivial. They are *not* a substitute for ImageNet, but they exercise
exactly the same code: folder-style labels, high-dimensional pixels, PCA, and $k$-NN.

In [ ]:
def find_class_dirs(root):
    # Return immediate subdirectories of root, descending through one wrapper folder.
    root = Path(root)
    if not root.is_dir():
        return []
    wrappers = {"train", "training", "images", "imagenet"}
    current = root
    for _ in range(3):
        subdirs = [p for p in current.iterdir() if p.is_dir()]
        if len(subdirs) == 1 and subdirs[0].name.lower() in wrappers:
            current = subdirs[0]
        else:
            break
    return [p for p in sorted(current.iterdir()) if p.is_dir()]


def load_image_folder_dataset(root, image_size=(32, 32), max_classes=6,
                              max_per_class=80, seed=0):
    # Load a small ImageFolder-style subset as flattened RGB float images.
    from PIL import Image

    class_dirs = []
    for directory in find_class_dirs(root):
        has_image = any(
            f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            for f in directory.iterdir()
        )
        if has_image:
            class_dirs.append(directory)

    if not class_dirs:
        raise FileNotFoundError(f"No class folders with images found under {root}")

    rng = np.random.default_rng(seed)
    chosen = rng.permutation(len(class_dirs))[:max_classes]
    class_dirs = [class_dirs[i] for i in chosen]

    X, y, names = [], [], []
    for label, directory in enumerate(class_dirs):
        files = sorted(
            f for f in directory.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        if len(files) > max_per_class:
            pick = rng.choice(len(files), size=max_per_class, replace=False)
            files = [files[i] for i in sorted(pick)]
        for path in files:
            with Image.open(path) as image:
                array = np.asarray(
                    image.convert("RGB").resize(image_size), dtype=np.float32
                ) / 255.0
            X.append(array.ravel())
            y.append(label)
        names.append(directory.name)

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64), names


def draw_synthetic_image(rng, label, size, colour):
    # Draw one class-specific shape on a noisy dark background.
    yy, xx = np.mgrid[0:size, 0:size]
    x = (xx - size / 2.0) / (size / 2.0)
    y = (yy - size / 2.0) / (size / 2.0)
    cx, cy = rng.uniform(-0.25, 0.25, size=2)
    radius = rng.uniform(0.35, 0.60)

    background = 0.15 + 0.10 * rng.random((size, size, 1))
    image = np.repeat(background, 3, axis=2)

    if label == 0:      # disk
        mask = (x - cx) ** 2 + (y - cy) ** 2 < radius ** 2
    elif label == 1:    # square
        mask = (np.abs(x - cx) < radius) & (np.abs(y - cy) < radius)
    elif label == 2:    # triangle
        top = cy - radius
        half_width = (y - top) / 2.0
        mask = (y > top) & (y < cy + radius) & (np.abs(x - cx) < half_width)
    elif label == 3:    # ring
        d2 = (x - cx) ** 2 + (y - cy) ** 2
        mask = (d2 > (0.55 * radius) ** 2) & (d2 < radius ** 2)
    elif label == 4:    # diagonal stripes
        phase = rng.uniform(0, 2 * np.pi)
        frequency = rng.uniform(6.0, 10.0)
        mask = np.sin(frequency * (x + y) + phase) > 0
    else:               # checkerboard
        frequency = rng.uniform(4.0, 7.0)
        phase = rng.uniform(0, np.pi)
        mask = np.sin(frequency * x + phase) * np.sin(frequency * y + phase) > 0

    # Per-image colour and brightness jitter make the classes overlap more.
    colour = np.clip(
        colour * rng.uniform(0.80, 1.10, size=3) * rng.uniform(0.70, 1.00),
        0.0,
        1.0,
    )
    image[mask] = colour

    # Add a neutral-coloured distractor patch in most images.
    if rng.random() < 0.85:
        dx, dy = rng.uniform(-0.6, 0.6, size=2)
        half_w, half_h = rng.uniform(0.06, 0.18, size=2)
        clutter = (np.abs(x - dx) < half_w) & (np.abs(y - dy) < half_h)
        image[clutter] = np.clip(0.40 + 0.15 * rng.random(3), 0.0, 1.0)

    image = image + rng.normal(0.0, 0.10, image.shape)
    return np.clip(image, 0.0, 1.0)


def make_synthetic_images(n_per_class=80, image_size=32, n_classes=6, seed=0):
    # Six classes of coloured shapes with the same tensor layout as ImageNet.
    rng = np.random.default_rng(seed)
    colours = np.array(
        [
            [0.90, 0.20, 0.20],
            [0.20, 0.40, 0.90],
            [0.20, 0.80, 0.30],
            [0.95, 0.85, 0.20],
            [0.70, 0.30, 0.90],
            [0.20, 0.85, 0.85],
        ]
    )
    names = [
        "red disk",
        "blue square",
        "green triangle",
        "yellow ring",
        "purple stripes",
        "cyan checkerboard",
    ]
    X, y = [], []
    for label in range(n_classes):
        for _ in range(n_per_class):
            X.append(draw_synthetic_image(rng, label, image_size, colours[label]).ravel())
            y.append(label)
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64), names

## 3. Load images

We try the real folder first and fall back to synthetic images only if it is absent or
unreadable. The printed source tells you which path was taken.

In [ ]:
try:
    if not IMAGENET_ROOT.is_dir():
        raise FileNotFoundError(f"{IMAGENET_ROOT} does not exist")
    X_img, y_img, img_class_names = load_image_folder_dataset(
        IMAGENET_ROOT, image_size=IMAGE_SIZE,
        max_classes=MAX_CLASSES, max_per_class=MAX_PER_CLASS,
        seed=RANDOM_STATE,
    )
    image_source = f"real images from {IMAGENET_ROOT}"
except Exception as exc:
    print(f"Using the synthetic fallback ({type(exc).__name__}: {exc})")
    X_img, y_img, img_class_names = make_synthetic_images(
        n_per_class=MAX_PER_CLASS, image_size=IMAGE_SIZE[0],
        n_classes=MAX_CLASSES, seed=RANDOM_STATE,
    )
    image_source = "synthetic shapes (fallback)"

print(f"Source: {image_source}")
print(f"Images: {X_img.shape}  (n, {IMAGE_SIZE[0]}*{IMAGE_SIZE[1]}*3)")
print(f"Classes: {len(img_class_names)} -> {img_class_names}")
print("Class counts:", np.bincount(y_img))

gallery = X_img[:12].reshape(-1, *IMAGE_SIZE, 3)
plot_image_gallery(
    gallery, y_img[:12], img_class_names, IMAGE_SIZE[:2],
    n=12, title=f"Training examples: {image_source}",
)

Each image is now a vector of $32 \times 32 \times 3 = 3{,}072$ numbers. Even at this
small resolution, one image has almost four times as many features as MNIST.

## 4. Tune and evaluate

The workflow is unchanged, but keep an eye on two numbers: accuracy and prediction time.
We reuse the same $k$ for a raw-pixel model and a PCA-40 model.

In [ ]:
X_im_train, X_im_test, y_im_train, y_im_test = train_test_split(
    X_img, y_img, test_size=0.25, stratify=y_img, random_state=RANDOM_STATE
)
print(f"Train: {X_im_train.shape}  Test: {X_im_test.shape}")

ks_img = [1, 3, 5, 7, 9, 15]
rows_img = tune_k(X_im_train, y_im_train, ks_img, cv_splits=3)
best_img = best_row(rows_img)

plot_k_curve(rows_img, "Images: choosing k with 3-fold CV")
for row in rows_img:
    print(f"k = {row['k']:2d}   CV accuracy = {row['mean']:.3f} +/- {row['std']:.3f}")
print(f"\nSelected k = {best_img['k']}")

In [ ]:
best_k_img = best_img["k"]
img_raw = fit_and_time(
    build_knn_pipeline(best_k_img),
    X_im_train, y_im_train, X_im_test, y_im_test,
)
img_pca = fit_and_time(
    build_knn_pipeline(best_k_img, n_components=40),
    X_im_train, y_im_train, X_im_test, y_im_test,
)

print(f"{'model':<22}{'test accuracy':>15}{'seconds':>10}{'ms/query':>10}")
print("-" * 57)
for name, result in [("raw pixels", img_raw), ("PCA-40", img_pca)]:
    print(f"{name:<22}{result['acc']:>15.4f}{result['seconds']:>10.2f}"
          f"{result['ms_per_query']:>10.3f}")

img_raw_acc, img_raw_ms = img_raw["acc"], img_raw["ms_per_query"]
img_pca_acc, img_pca_ms = img_pca["acc"], img_pca["ms_per_query"]

## 5. Why raw-pixel $k$-NN does not scale to ImageNet

Two effects appear as the dimension grows.

**1. Distances concentrate.** A useful diagnostic is the ratio between the nearest and
the farthest distance from a query. If that ratio is close to one, "nearest" is barely
nearer than "farthest", and neighbour voting loses its meaning.

**2. Prediction cost explodes.** Brute force compares each query with every stored
training image in every pixel. For full ImageNet with raw pixels this is hopeless.

The next cell measures distance concentration on these images, then on uniformly random
points as a baseline. The baseline shows the pure effect of dimension; real data can do
better when it lies near a low-dimensional manifold.

In [ ]:
def distance_concentration(X_query, X_reference, max_queries=40, random_state=42):
    # Mean ratio of nearest to farthest distance over a random sample of queries.
    rng = np.random.default_rng(random_state)
    idx = rng.permutation(len(X_query))[:max_queries]
    ratios = []
    for query in X_query[idx]:
        distances = np.linalg.norm(X_reference - query, axis=1)
        ratios.append(distances.min() / distances.max())
    return float(np.mean(ratios))


ratio = distance_concentration(X_im_test, X_im_train)
print(f"Images  nearest/farthest distance ratio = {ratio:.3f}")

print("\nUniform random baseline (40 queries, 200 reference points):")
for d in [4, 64, 3072]:
    rng = np.random.default_rng(RANDOM_STATE)
    A = rng.random((40, d))
    B = rng.random((200, d))
    baseline = distance_concentration(A, B, max_queries=40)
    print(f"d = {d:4d}   nearest/farthest distance ratio = {baseline:.3f}")

For uniformly random points the ratio climbs toward one as the dimension grows: about
0.14 at $d=4$, 0.68 at $d=64$, and 0.95 at $d=3072$. In that regime "nearest" is almost
meaningless.

Real data is not uniformly random. These small images have a ratio well below the random
baseline because simple shapes lie near a low-dimensional manifold, so genuine neighbours
still exist. Natural ImageNet images at full resolution are much less forgiving. Pixel
distances concentrate strongly, and even where they do not, the brute-force cost is
prohibitive.

Real ImageNet pipelines therefore do **not** run $k$-NN on raw pixels. They

- extract features with a pretrained CNN or vision transformer,
- optionally reduce them with PCA or a learned projection,
- then run $k$-NN or a linear classifier on the embeddings, or
- use approximate nearest-neighbour indexes (FAISS, HNSW, Annoy) for retrieval.

The k-NN *idea* survives; the raw pixel representation does not.

## 6. Confusion matrix and errors

In [ ]:
if img_pca["acc"] >= img_raw["acc"]:
    img_best, img_best_name = img_pca, "PCA-40 + k-NN"
else:
    img_best, img_best_name = img_raw, "raw pixels + k-NN"

img_pred = img_best["y_pred"]
print(f"Diagnosing: {img_best_name}")
print(f"Test accuracy: {img_best['acc']:.4f}\n")
print(classification_report(
    y_im_test, img_pred, target_names=img_class_names, digits=3, zero_division=0,
))

cm_img = confusion_matrix(y_im_test, img_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm_img, display_labels=img_class_names).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_title(f"Image subset confusion matrix ({img_best_name}, k = {best_k_img})")
plt.show()

In [ ]:
wrong_img = np.where(img_pred != y_im_test)[0]
print(f"Misclassified: {len(wrong_img)} / {len(y_im_test)} images")

if len(wrong_img):
    n_show = min(8, len(wrong_img))
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, idx in zip(axes.ravel(), wrong_img[:n_show]):
        ax.imshow(X_im_test[idx].reshape(*IMAGE_SIZE, 3))
        ax.set_title(
            f"true: {img_class_names[y_im_test[idx]]}\n"
            f"pred: {img_class_names[img_pred[idx]]}",
            fontsize=8, color="darkred",
        )
    fig.suptitle("Misclassified images")
    fig.tight_layout()
    plt.show()
else:
    print("No errors on this small test subset.")

**Takeaways**

- the same $k$-NN code runs on images, but the representation matters more than the
  classifier;
- $32 \times 32$ pixels is already 3,072 features, and real images are far larger;
- PCA gives a large speed-up, but it is a linear projection and cannot fix a bad
  representation;
- for real work, use CNN/transformer embeddings and approximate nearest-neighbour search.

## What to try next

1. **Resolution.** Change `IMAGE_SIZE` to $(16, 16)$ and $(64, 64)$. How do accuracy and
   prediction time change?
2. **Distance concentration.** Extend the experiment to Gaussian random data in
   dimensions 2, 10, 50, 200, and 1000. Compare with the image curve.
3. **Cosine distance.** Add `metric="cosine"` to the classifier pipeline and compare with
   Euclidean distance.
4. **PCA variance.** Plot the cumulative explained variance and find the number of
   components needed for 90%.
5. **Real ImageNet.** If you have a local copy, load 10 classes and compare raw pixels
   with features from a pretrained network. Use a nearest-neighbour index if the sample
   is large.
6. **More classes.** Raise `MAX_CLASSES` and `MAX_PER_CLASS`. At what point does the
   brute-force search become too slow for interactive use?

## Further reading

- J. Deng et al., "ImageNet: A Large-Scale Hierarchical Image Database", *CVPR*, 2009.
- A. Krizhevsky, I. Sutskever, and G. E. Hinton, "ImageNet Classification with Deep
  Convolutional Neural Networks", *NeurIPS*, 2012.
- J. Johnson, M. Douze, and H. Jegou, "Billion-scale similarity search with GPUs"
  (FAISS), *IEEE Transactions on Big Data*, 2019.
- scikit-learn user guide, *Nearest Neighbors* and *Approximate Nearest Neighbors*.